# Azure Content Understanding - API Testing Guide (C#)

This notebook replicates all the API tests from the `.http` test files using **C#** with the .NET Interactive (Polyglot Notebooks) kernel.

**Documentation:**
- [Overview](https://learn.microsoft.com/en-us/azure/ai-services/content-understanding/overview)
- [REST API Reference](https://learn.microsoft.com/en-us/rest/api/contentunderstanding/operation-groups)

## Table of Contents
1. **Setup** – NuGet packages, load credentials, create HttpClient
2. **Quick Start** – Check defaults, layout extraction, invoice analysis
3. **Section 1** – Content Extraction (no LLM required)
4. **Section 2** – Domain-Specific Analyzers (invoice, document)
5. **Section 3** – RAG Analyzers (documentSearch, imageSearch, etc.)
6. **Section 4** – Custom Analyzers with Field Extraction
7. **Section 5** – Custom Video Analysis with Field Extraction
8. **Section 6** – Analyzer Management (List, Copy, Delete)
9. **Bonus** – Custom Analyzer with allowReplace

## Setup

Authenticate via **Microsoft Entra ID** using `DefaultAzureCredential`.

**Before running this notebook:**
1. Run `az login` in your terminal (authenticates Azure CLI)
2. Ensure you have the **Cognitive Services User** role on the CU resource (ask your infra team to assign it)

Set the following environment variables (or use a `.env` file with `dotnet-env`):
```
ENDPOINT_URL=https://your-resource.cognitiveservices.azure.com
API_VERSION=2025-11-01
AOAI_CONNECTION=your-aoai-connection-name
GPT41_DEPLOYMENT=gpt-41
GPT41_MINI_DEPLOYMENT=gpt-41-mini
EMBEDDING_ADA_DEPLOYMENT=text-embedding-ada-002
EMBEDDING_3LARGE_DEPLOYMENT=text-embedding-3-large
```

> **Note:** No API key is needed — authentication uses your Azure CLI login token via `DefaultAzureCredential`.

**Prerequisites:** .NET 8+ SDK, VS Code with [Polyglot Notebooks](https://marketplace.visualstudio.com/items?itemName=ms-dotnettools.dotnet-interactive-vscode) extension, Azure CLI (`az login`).

In [ ]:
// NuGet packages (System.Net.Http and System.Text.Json are built-in to .NET Interactive)
#r "nuget: Microsoft.Extensions.Configuration, 9.*"
#r "nuget: Microsoft.Extensions.Configuration.EnvironmentVariables, 9.*"
#r "nuget: Azure.Identity, 1.*"

Installed Packages Microsoft.Extensions.Configuration, 9.0.14 Microsoft.Extensions.Configuration.EnvironmentVariables, 9.0.14

In [ ]:
using System;
using System.Net.Http;
using System.Net.Http.Headers;
using System.Text;
using System.Text.Json;
using System.Threading.Tasks;
using Microsoft.Extensions.Configuration;
using Azure.Identity;
using Azure.Core;

// Load configuration from environment variables
var config = new ConfigurationBuilder()
    .AddEnvironmentVariables()
    .Build();

var ENDPOINT            = (config["ENDPOINT_URL"] ?? throw new Exception("ENDPOINT_URL environment variable must be set")).TrimEnd('/');
var API_VERSION         = config["API_VERSION"] ?? "2025-11-01";
var AOAI_CONNECTION     = config["AOAI_CONNECTION"] ?? throw new Exception("AOAI_CONNECTION environment variable must be set");
var GPT41_DEPLOYMENT    = config["GPT41_DEPLOYMENT"] ?? "gpt-41";
var GPT41_MINI_DEPLOYMENT       = config["GPT41_MINI_DEPLOYMENT"] ?? "gpt-41-mini";
var EMBEDDING_ADA_DEPLOYMENT    = config["EMBEDDING_ADA_DEPLOYMENT"] ?? "text-embedding-ada-002";
var EMBEDDING_3LARGE_DEPLOYMENT = config["EMBEDDING_3LARGE_DEPLOYMENT"] ?? "text-embedding-3-large";

// Authenticate via Entra ID (DefaultAzureCredential chains: AzureCLI → InteractiveBrowser → ManagedIdentity)
var credential = new DefaultAzureCredential();
var tokenResult = await credential.GetTokenAsync(new TokenRequestContext(new[] { "https://cognitiveservices.azure.com/.default" }));

var client = new HttpClient();
client.DefaultRequestHeaders.Authorization = new AuthenticationHeaderValue("Bearer", tokenResult.Token);

Console.WriteLine($"Endpoint: {ENDPOINT}");
Console.WriteLine($"API Version: {API_VERSION}");
Console.WriteLine($"AOAI Connection: {AOAI_CONNECTION}");
Console.WriteLine($"Token expires: {tokenResult.ExpiresOn:u} (re-run this cell to refresh)");
Console.WriteLine("Credentials loaded successfully (Entra ID).");

Error: System.Exception: API_KEY environment variable must be set
   at Submission#2.<<Initialize>>d__0.MoveNext()
--- End of stack trace from previous location ---
   at Microsoft.CodeAnalysis.Scripting.ScriptExecutionState.RunSubmissionsAsync[TResult](ImmutableArray`1 precedingExecutors, Func`2 currentExecutor, StrongBox`1 exceptionHolderOpt, Func`2 catchExceptionOpt, CancellationToken cancellationToken)

In [ ]:
// JSON serializer options for pretty-printing
var jsonOptions = new JsonSerializerOptions { WriteIndented = true };

void Pretty(JsonElement element)
{
    Console.WriteLine(JsonSerializer.Serialize(element, jsonOptions));
}

void Pretty(object obj)
{
    Console.WriteLine(JsonSerializer.Serialize(obj, jsonOptions));
}

async Task<JsonDocument> AnalyzeAndPollAsync(string analyzerId, object payload, int pollInterval = 5, int maxWait = 300)
{
    var url = $"{ENDPOINT}/contentunderstanding/analyzers/{analyzerId}:analyze?api-version={API_VERSION}";
    var content = new StringContent(JsonSerializer.Serialize(payload), Encoding.UTF8, "application/json");
    var resp = await client.PostAsync(url, content);
    resp.EnsureSuccessStatusCode();

    var operationDoc = JsonDocument.Parse(await resp.Content.ReadAsStringAsync());
    var operationId = operationDoc.RootElement.GetProperty("id").GetString();
    var status = operationDoc.RootElement.TryGetProperty("status", out var s) ? s.GetString() : "unknown";
    Console.WriteLine($"Operation started: {operationId}");
    Console.WriteLine($"Status: {status}");

    var resultUrl = $"{ENDPOINT}/contentunderstanding/analyzerResults/{operationId}?api-version={API_VERSION}";
    var elapsed = 0;
    JsonDocument result = null;

    while (elapsed < maxWait)
    {
        await Task.Delay(pollInterval * 1000);
        elapsed += pollInterval;

        var resultResp = await client.GetAsync(resultUrl);
        resultResp.EnsureSuccessStatusCode();
        result = JsonDocument.Parse(await resultResp.Content.ReadAsStringAsync());
        status = result.RootElement.TryGetProperty("status", out var st) ? st.GetString() : "unknown";
        Console.WriteLine($"  [{elapsed}s] Status: {status}");

        if (status == "Succeeded" || status == "succeeded" || status == "Failed" || status == "failed")
            return result;
    }

    Console.WriteLine($"Timed out after {maxWait}s. Returning last response.");
    return result;
}

---
## Quick Start: Get Started in 3 Steps

1. Check model deployments (GPT-4.1, GPT-4.1-mini, text-embedding-ada-002)
2. Try content extraction (no LLM required)
3. Try a domain-specific analyzer (invoice)

### Step 1: Check current model deployment settings

In [ ]:
var resp = await client.GetAsync($"{ENDPOINT}/contentunderstanding/defaults?api-version={API_VERSION}");
resp.EnsureSuccessStatusCode();
var doc = JsonDocument.Parse(await resp.Content.ReadAsStringAsync());
Pretty(doc.RootElement);

### Step 1b: Configure model deployments (if needed)

Only required if using BYOC (Bring Your Own Compute) with custom Azure OpenAI deployments.  
Update the deployment names below to match your setup, then run the cell.

In [ ]:
var patchBody = new
{
    modelDeployments = new Dictionary<string, string>
    {
        ["gpt-4.1"] = $"{AOAI_CONNECTION}/{GPT41_DEPLOYMENT}",
        ["text-embedding-ada-002"] = $"{AOAI_CONNECTION}/{EMBEDDING_ADA_DEPLOYMENT}",
        ["text-embedding-3-large"] = $"{AOAI_CONNECTION}/{EMBEDDING_3LARGE_DEPLOYMENT}",
        ["gpt-4.1-mini"] = $"{AOAI_CONNECTION}/{GPT41_MINI_DEPLOYMENT}",
        ["prebuilt-analyzer-completion"] = $"{AOAI_CONNECTION}/{GPT41_DEPLOYMENT}",
        ["prebuilt-analyzer-embedding"] = $"{AOAI_CONNECTION}/{EMBEDDING_3LARGE_DEPLOYMENT}",
        ["prebuilt-analyzer-completion-mini"] = $"{AOAI_CONNECTION}/{GPT41_MINI_DEPLOYMENT}"
    }
};

var request = new HttpRequestMessage(HttpMethod.Patch, $"{ENDPOINT}/contentunderstanding/defaults?api-version={API_VERSION}")
{
    Content = new StringContent(JsonSerializer.Serialize(patchBody), Encoding.UTF8, "application/json")
};
var resp = await client.SendAsync(request);
resp.EnsureSuccessStatusCode();
var doc = JsonDocument.Parse(await resp.Content.ReadAsStringAsync());
Pretty(doc.RootElement);

### Step 2: Content extraction with prebuilt-layout

This analyzer does **NOT** require LLM models – works immediately.  
Extracts text, tables, figures, sections, hyperlinks, and annotations.

In [ ]:
var result = await AnalyzeAndPollAsync("prebuilt-layout", new
{
    inputs = new[] { new { url = "https://github.com/Azure-Samples/cognitive-services-sample-data-files/raw/master/ComputerVision/Images/printed_text.jpg" } }
});
Pretty(result.RootElement);

### Step 3: Domain-specific analyzer – prebuilt-invoice

Extracts vendor, items, totals, dates from invoices.  
Uses the LLM in default/models for field extraction.

In [ ]:
var result = await AnalyzeAndPollAsync("prebuilt-invoice", new
{
    inputs = new[] { new { url = "https://github.com/Azure-Samples/azure-ai-content-understanding-python/raw/refs/heads/main/data/invoice.pdf" } }
});
Pretty(result.RootElement);

---
## Section 1: Content Extraction (No LLM Required)

These analyzers do **NOT** require LLM or embedding models. They work immediately after resource creation.

- **prebuilt-read** – Basic OCR (words, paragraphs, formulas, barcodes)
- **prebuilt-layout** – Advanced OCR with layout (tables, figures, sections)

### 1.1 prebuilt-read: Basic OCR

Extracts words, paragraphs, formulas, barcodes without layout analysis.

In [ ]:
var result = await AnalyzeAndPollAsync("prebuilt-read", new
{
    inputs = new[] { new { url = "https://github.com/Azure-Samples/cognitive-services-sample-data-files/raw/master/ComputerVision/Images/handwritten_text.jpg" } }
});
Pretty(result.RootElement);

### 1.2 prebuilt-layout: OCR with layout analysis

Extracts text, tables, figures, sections, hyperlinks, annotations.

In [ ]:
var result = await AnalyzeAndPollAsync("prebuilt-layout", new
{
    inputs = new[] { new { url = "https://github.com/Azure-Samples/cognitive-services-sample-data-files/raw/master/ComputerVision/Images/printed_text.jpg" } }
});
Pretty(result.RootElement);

---
## Section 2: Domain-Specific Analyzers

Preconfigured analyzers for common document types. Powered by rich knowledge bases of real-world examples.

Available analyzers:
- **Financial:** prebuilt-invoice, prebuilt-receipt, prebuilt-creditCard
- **Identity:** prebuilt-idDocument, prebuilt-healthInsuranceCard.us
- **Tax (US):** prebuilt-tax.us.1040, prebuilt-tax.us.w2, prebuilt-tax.us.1099*
- **Mortgage (US):** prebuilt-mortgage.us.1003, prebuilt-mortgage.us.1004
- **Other:** prebuilt-utilityBill, prebuilt-payStub.us, prebuilt-contract

### 2.1 prebuilt-invoice

In [ ]:
var result = await AnalyzeAndPollAsync("prebuilt-invoice", new
{
    inputs = new[] { new { url = "https://github.com/Azure-Samples/azure-ai-content-understanding-python/raw/refs/heads/main/data/invoice.pdf" } }
});
Pretty(result.RootElement);

### 2.2 prebuilt-document

General-purpose document analysis with tables, key-value pairs, entities.  
Works with any document type – no domain-specific training needed.

In [ ]:
var result = await AnalyzeAndPollAsync("prebuilt-document", new
{
    inputs = new[] { new { url = "https://github.com/Azure-Samples/azure-ai-content-understanding-python/raw/refs/heads/main/data/mixed_financial_docs.pdf" } }
});
Pretty(result.RootElement);

---
## Section 3: RAG Analyzers (Requires LLM + Embeddings)

Optimized for RAG scenarios with semantic analysis and markdown extraction.

- **prebuilt-documentSearch** – Document ingestion with summaries
- **prebuilt-imageSearch** – Image analysis with descriptions
- **prebuilt-audioSearch** – Audio transcription with summaries
- **prebuilt-videoSearch** – Video segmentation with transcripts

> **Tip:** Use the `range` parameter to limit analysis to specific pages (e.g., `"1-3"`, `"1,5,7"`).

### 3.1 prebuilt-documentSearch

Extracts content, tables, figures with descriptions, chart.js/mermaid.js output.  
Generates document summaries, captures annotations.

In [ ]:
var result = await AnalyzeAndPollAsync("prebuilt-documentSearch", new
{
    inputs = new[] { new { url = "https://raw.githubusercontent.com/Azure-Samples/azure-search-sample-data/refs/heads/main/sustainable-ai-pdf/Accelerating-Sustainability-with-AI-2025.pdf", range = "1-3" } }
});
Pretty(result.RootElement);

### 3.1b Get RAG analyzer definition

In [ ]:
var resp = await client.GetAsync($"{ENDPOINT}/contentunderstanding/analyzers/prebuilt-documentSearch?api-version={API_VERSION}");
resp.EnsureSuccessStatusCode();
var doc = JsonDocument.Parse(await resp.Content.ReadAsStringAsync());
Pretty(doc.RootElement);

---
## Section 4: Custom Analyzers with Field Extraction

**Field Extraction** is the core value of custom analyzers. Define a field schema to extract specific structured data.

Field extraction methods:
- **extract** – Extract existing data from the content (e.g., policy number)
- **classify** – Classify content into predefined categories (e.g., loss type)
- **generate** – Generate new insights from the content (e.g., summaries)

### 4.1 Create custom invoice analyzer with field extraction

In [ ]:
var customInvoiceSchema = new
{
    baseAnalyzerId = "prebuilt-document",
    analyzerId = "custom_invoice",
    models = new { completion = "gpt-4.1", embedding = "text-embedding-ada-002" },
    fieldSchema = new
    {
        fields = new Dictionary<string, object>
        {
            ["InvoiceNumber"] = new { type = "string", method = "extract", description = "The invoice number (e.g., INV-100)" },
            ["InvoiceDate"] = new { type = "string", method = "extract", description = "The invoice date" },
            ["CustomerName"] = new { type = "string", method = "extract", description = "The customer name or company name" },
            ["TotalAmount"] = new { type = "number", method = "extract", description = "The total amount due on the invoice" },
            ["Subtotal"] = new { type = "number", method = "extract", description = "The subtotal before taxes" },
            ["SalesTax"] = new { type = "number", method = "extract", description = "The sales tax amount" },
            ["LineItems"] = new
            {
                type = "array",
                method = "extract",
                description = "Individual line items from the invoice",
                items = new
                {
                    type = "object",
                    properties = new Dictionary<string, object>
                    {
                        ["Date"] = new { type = "string", description = "Date of the service or item" },
                        ["Description"] = new { type = "string", description = "Description of the service or item" },
                        ["Quantity"] = new { type = "number", description = "Quantity or hours" },
                        ["Price"] = new { type = "number", description = "Unit price" },
                        ["Amount"] = new { type = "number", description = "Line item total amount" }
                    }
                }
            }
        },
        definitions = new { }
    },
    omitContent = true
};

var content = new StringContent(JsonSerializer.Serialize(customInvoiceSchema), Encoding.UTF8, "application/json");
var resp = await client.PutAsync($"{ENDPOINT}/contentunderstanding/analyzers/custom_invoice?api-version={API_VERSION}", content);
Console.WriteLine($"Status: {(int)resp.StatusCode}");
var doc = JsonDocument.Parse(await resp.Content.ReadAsStringAsync());
Pretty(doc.RootElement);

### Get custom invoice analyzer definition

In [ ]:
var resp = await client.GetAsync($"{ENDPOINT}/contentunderstanding/analyzers/custom_invoice?api-version={API_VERSION}");
resp.EnsureSuccessStatusCode();
var doc = JsonDocument.Parse(await resp.Content.ReadAsStringAsync());
Pretty(doc.RootElement);

### 4.2 Test the custom invoice analyzer

In [ ]:
var result = await AnalyzeAndPollAsync("custom_invoice", new
{
    inputs = new[] { new { url = "https://github.com/Azure-Samples/azure-ai-content-understanding-python/raw/refs/heads/main/data/invoice.pdf" } }
});
Pretty(result.RootElement);

---
## Section 5: Custom Video Analysis with Field Extraction

Video analyzers demonstrate advanced field extraction with nested structures.  
Extract segments, scenes, timestamps, and generated insights from video content.

> **Note:** Video processing takes 1–2× the video duration. A 2-minute video may take 2–4 minutes to complete.

### 5.1 Create custom video analyzer (dynamic chaptering)

In [ ]:
var videoSchema = new
{
    description = "Dynamic video chaptering with scene detection",
    scenario = "videoShot",
    baseAnalyzerId = "prebuilt-video",
    models = new { completion = "gpt-4.1" },
    config = new
    {
        returnDetails = true,
        enableSegmentation = true,
        locales = new[] { "en-US" }
    },
    fieldSchema = new
    {
        name = "Content Understanding - Dynamic Chaptering",
        fields = new Dictionary<string, object>
        {
            ["Segments"] = new
            {
                type = "array",
                items = new
                {
                    type = "object",
                    properties = new Dictionary<string, object>
                    {
                        ["SegmentId"] = new { type = "string" },
                        ["SegmentType"] = new { type = "string", method = "generate", description = "The short title or a short summary of the story or chapter." },
                        ["Scenes"] = new
                        {
                            type = "array",
                            items = new
                            {
                                type = "object",
                                properties = new Dictionary<string, object>
                                {
                                    ["Description"] = new { type = "string", method = "generate", description = "A five-word description of the scene." },
                                    ["StartTimestamp"] = new { type = "string", description = "the start timestamp of the scene" },
                                    ["EndTimestamp"] = new { type = "string", description = "the end timestamp of the scene" }
                                }
                            }
                        }
                    }
                }
            }
        }
    }
};

var content = new StringContent(JsonSerializer.Serialize(videoSchema), Encoding.UTF8, "application/json");
var resp = await client.PutAsync($"{ENDPOINT}/contentunderstanding/analyzers/video_scenes_list?api-version={API_VERSION}", content);
Console.WriteLine($"Status: {(int)resp.StatusCode}");
var doc = JsonDocument.Parse(await resp.Content.ReadAsStringAsync());
Pretty(doc.RootElement);

### Get video analyzer definition

In [ ]:
var resp = await client.GetAsync($"{ENDPOINT}/contentunderstanding/analyzers/video_chaptering?api-version={API_VERSION}");
resp.EnsureSuccessStatusCode();
var doc = JsonDocument.Parse(await resp.Content.ReadAsStringAsync());
Pretty(doc.RootElement);

### 5.2 Test video analyzer

> Video analysis may take several minutes depending on video length.

In [ ]:
var result = await AnalyzeAndPollAsync("video_chaptering", new
{
    inputs = new[] { new { url = "https://github.com/Azure-Samples/azure-ai-content-understanding-python/raw/refs/heads/main/data/FlightSimulator.mp4" } }
}, pollInterval: 15, maxWait: 600);
Pretty(result.RootElement);

---
## Section 6: Analyzer Management (List, Copy, Delete)

### 6.1 List all available analyzers

In [ ]:
var resp = await client.GetAsync($"{ENDPOINT}/contentunderstanding/analyzers?api-version={API_VERSION}");
resp.EnsureSuccessStatusCode();
var doc = JsonDocument.Parse(await resp.Content.ReadAsStringAsync());
Pretty(doc.RootElement);

### 6.2 List only prebuilt analyzers

In [ ]:
var resp = await client.GetAsync($"{ENDPOINT}/contentunderstanding/analyzers?api-version={API_VERSION}&$filter=startswith(analyzerId,'prebuilt-')");
resp.EnsureSuccessStatusCode();
var doc = JsonDocument.Parse(await resp.Content.ReadAsStringAsync());
Pretty(doc.RootElement);

### 6.3 Get specific analyzer definition (prebuilt-invoice)

In [ ]:
var resp = await client.GetAsync($"{ENDPOINT}/contentunderstanding/analyzers/prebuilt-invoice?api-version={API_VERSION}");
resp.EnsureSuccessStatusCode();
var doc = JsonDocument.Parse(await resp.Content.ReadAsStringAsync());
Pretty(doc.RootElement);

### 6.4 Copy analyzer within same resource

Useful for versioning or creating variations.

In [ ]:
var copyBody = new { sourceAnalyzerId = "video_scenes_list" };
var content = new StringContent(JsonSerializer.Serialize(copyBody), Encoding.UTF8, "application/json");
var resp = await client.PostAsync($"{ENDPOINT}/contentunderstanding/analyzers/video_scenes_list-v2:copy?api-version={API_VERSION}", content);
Console.WriteLine($"Status: {(int)resp.StatusCode}");
var doc = JsonDocument.Parse(await resp.Content.ReadAsStringAsync());
Pretty(doc.RootElement);

### 6.5 Copy prebuilt to create stable version

Lock a prebuilt analyzer definition to prevent changes across API versions.

In [ ]:
var copyBody = new { sourceAnalyzerId = "prebuilt-invoice" };
var content = new StringContent(JsonSerializer.Serialize(copyBody), Encoding.UTF8, "application/json");
var resp = await client.PostAsync($"{ENDPOINT}/contentunderstanding/analyzers/my-invoice:copy?api-version={API_VERSION}", content);
Console.WriteLine($"Status: {(int)resp.StatusCode}");
var doc = JsonDocument.Parse(await resp.Content.ReadAsStringAsync());
Pretty(doc.RootElement);

### 6.6 Delete custom analyzer

In [ ]:
var resp = await client.DeleteAsync($"{ENDPOINT}/contentunderstanding/analyzers/my-invoice?api-version={API_VERSION}");
Console.WriteLine($"Status: {(int)resp.StatusCode}");
var body = await resp.Content.ReadAsStringAsync();
if (!string.IsNullOrWhiteSpace(body))
{
    var doc = JsonDocument.Parse(body);
    Pretty(doc.RootElement);
}
else
{
    Console.WriteLine("Analyzer deleted successfully.");
}

---
## Bonus: Custom Analyzer with allowReplace

The `allowReplace=true` query parameter enables creating or replacing an existing analyzer with the same ID.  
This is useful for updating analyzer schemas without conflicts.

### Create custom claims analyzer (with allowReplace=true)

In [ ]:
var claimsSchemaV1 = new
{
    description = "Custom analyzer for insurance claims processing",
    tags = new Dictionary<string, string>
    {
        ["version"] = "1.0",
        ["category"] = "claims"
    },
    baseAnalyzerId = "prebuilt-document",
    models = new { completion = "gpt-4.1", embedding = "text-embedding-3-large" },
    fieldSchema = new
    {
        name = "Insurance Claims Schema",
        description = "Extract key information from insurance claims",
        fields = new Dictionary<string, object>
        {
            ["ClaimNumber"] = new { type = "string", method = "extract", description = "The unique claim reference number" },
            ["ClaimDate"] = new { type = "date", method = "extract", description = "The date the claim was filed" },
            ["PolicyHolder"] = new { type = "string", method = "extract", description = "Name of the policy holder" },
            ["PolicyNumber"] = new { type = "string", method = "extract", description = "The insurance policy number" },
            ["LossType"] = new
            {
                type = "string", method = "classify", description = "Category of loss",
                @enum = new[] { "property", "auto", "health", "liability", "other" }
            },
            ["ClaimAmount"] = new { type = "number", method = "extract", description = "The amount claimed" },
            ["ClaimSummary"] = new { type = "string", method = "generate", description = "AI-generated summary of the claim details" },
            ["DocumentedExpenses"] = new
            {
                type = "array", method = "extract",
                description = "List of expenses documented in the claim",
                items = new
                {
                    type = "object",
                    properties = new Dictionary<string, object>
                    {
                        ["Date"] = new { type = "date", description = "Date of the expense" },
                        ["Category"] = new { type = "string", description = "Category of expense" },
                        ["Amount"] = new { type = "number", description = "Amount of the expense" },
                        ["Description"] = new { type = "string", description = "Details about the expense" }
                    }
                }
            }
        },
        definitions = new { }
    },
    config = new
    {
        enableOcr = true,
        enableLayout = true,
        enableFormula = false,
        returnDetails = true,
        omitContent = false
    }
};

var content = new StringContent(JsonSerializer.Serialize(claimsSchemaV1), Encoding.UTF8, "application/json");
var resp = await client.PutAsync($"{ENDPOINT}/contentunderstanding/analyzers/custom_claims_analyzer?api-version={API_VERSION}&allowReplace=true", content);
Console.WriteLine($"Status: {(int)resp.StatusCode}");
var doc = JsonDocument.Parse(await resp.Content.ReadAsStringAsync());
Pretty(doc.RootElement);

### Get custom claims analyzer definition

In [ ]:
var resp = await client.GetAsync($"{ENDPOINT}/contentunderstanding/analyzers/custom_claims_analyzer?api-version={API_VERSION}");
resp.EnsureSuccessStatusCode();
var doc = JsonDocument.Parse(await resp.Content.ReadAsStringAsync());
Pretty(doc.RootElement);

### Test the custom claims analyzer

In [ ]:
var result = await AnalyzeAndPollAsync("custom_claims_analyzer", new
{
    inputs = new[] { new { url = "https://github.com/Azure-Samples/azure-ai-content-understanding-python/raw/refs/heads/main/data/invoice.pdf" } }
});
Pretty(result.RootElement);

### Update claims analyzer to v2 (with allowReplace=true)

Adds `ClaimStatus`, `AdjusterName`, `RiskAssessment` fields and `Verified` to expenses.

In [ ]:
var claimsSchemaV2 = new
{
    description = "Updated custom analyzer for insurance claims processing - v2",
    tags = new Dictionary<string, string>
    {
        ["version"] = "2.0",
        ["category"] = "claims",
        ["lastUpdated"] = "2025-01-29"
    },
    baseAnalyzerId = "prebuilt-document",
    models = new { completion = "gpt-4.1", embedding = "text-embedding-3-large" },
    fieldSchema = new
    {
        name = "Insurance Claims Schema v2",
        description = "Enhanced schema for insurance claims with additional fields",
        fields = new Dictionary<string, object>
        {
            ["ClaimNumber"] = new { type = "string", method = "extract", description = "The unique claim reference number" },
            ["ClaimDate"] = new { type = "date", method = "extract", description = "The date the claim was filed" },
            ["PolicyHolder"] = new { type = "string", method = "extract", description = "Name of the policy holder" },
            ["PolicyNumber"] = new { type = "string", method = "extract", description = "The insurance policy number" },
            ["LossType"] = new
            {
                type = "string", method = "classify", description = "Category of loss",
                @enum = new[] { "property", "auto", "health", "liability", "other" }
            },
            ["ClaimAmount"] = new { type = "number", method = "extract", description = "The amount claimed" },
            ["ClaimStatus"] = new
            {
                type = "string", method = "classify", description = "Current status of the claim",
                @enum = new[] { "pending", "approved", "denied", "settled" }
            },
            ["AdjusterName"] = new { type = "string", method = "extract", description = "Name of the assigned claims adjuster" },
            ["ClaimSummary"] = new { type = "string", method = "generate", description = "AI-generated summary of the claim details" },
            ["RiskAssessment"] = new { type = "string", method = "generate", description = "AI-generated risk assessment based on claim details" },
            ["DocumentedExpenses"] = new
            {
                type = "array", method = "extract",
                description = "List of expenses documented in the claim",
                items = new
                {
                    type = "object",
                    properties = new Dictionary<string, object>
                    {
                        ["Date"] = new { type = "date", description = "Date of the expense" },
                        ["Category"] = new { type = "string", description = "Category of expense" },
                        ["Amount"] = new { type = "number", description = "Amount of the expense" },
                        ["Description"] = new { type = "string", description = "Details about the expense" },
                        ["Verified"] = new { type = "boolean", description = "Whether the expense has been verified" }
                    }
                }
            }
        },
        definitions = new { }
    },
    config = new
    {
        enableOcr = true,
        enableLayout = true,
        enableFormula = false,
        returnDetails = true,
        omitContent = false,
        estimateFieldSourceAndConfidence = true
    }
};

var content = new StringContent(JsonSerializer.Serialize(claimsSchemaV2), Encoding.UTF8, "application/json");
var resp = await client.PutAsync($"{ENDPOINT}/contentunderstanding/analyzers/custom_claims_analyzer?api-version={API_VERSION}&allowReplace=true", content);
Console.WriteLine($"Status: {(int)resp.StatusCode}");
var doc = JsonDocument.Parse(await resp.Content.ReadAsStringAsync());
Pretty(doc.RootElement);

### Create analyzer without allowReplace (strict creation)

Will return 201 if successful, or fail with a conflict error if the analyzer already exists.

In [ ]:
var simpleSchema = new
{
    description = "Custom analyzer without allow replace",
    baseAnalyzerId = "prebuilt-document",
    models = new { completion = "gpt-4.1" },
    fieldSchema = new
    {
        name = "Simple Schema",
        fields = new Dictionary<string, object>
        {
            ["Title"] = new { type = "string", method = "extract", description = "Document title" }
        },
        definitions = new { }
    }
};

var content = new StringContent(JsonSerializer.Serialize(simpleSchema), Encoding.UTF8, "application/json");
var resp = await client.PutAsync($"{ENDPOINT}/contentunderstanding/analyzers/custom_new_analyzer?api-version={API_VERSION}", content);
Console.WriteLine($"Status: {(int)resp.StatusCode}");
var doc = JsonDocument.Parse(await resp.Content.ReadAsStringAsync());
Pretty(doc.RootElement);